In [2]:
import pyterrier as pt
import requests
import pandas as pd
import pickle
import nltk
import numpy as np
%pip install xlrd

Note: you may need to restart the kernel to use updated packages.


In [3]:
df_alma = pd.read_excel('alma_beck.xlsx', sheet_name='getFullTxt')

In [4]:
df_alma

,LOCALIZED,ISSN,ISSN.1,ISSN.2,ISBN1,ISBN2,ISBN3,OTHER_SYSTEM_NUMBER1,OTHER_SYSTEM_NUMBER2,OTHER_SYSTEM_NUMBER3,...,LICENSE,LICENSE_NAME,COLLECTION_LICENSE,COLLECTION_LICENSE_NAME,PDA,EBA,PO_LINE,LANGUAGE,COUNTER_PLATFORM,NOTES
0,N,NaN,NaN,NaN,9783406751332,9.783407e+12,NaN,(ALEPH)020383966EDU01,(DE-465)LT001916108,LT001916108,...,NaN,NaN,NaN,NaN,NaN,NaN,UB23-25962,ger,NaN,NaN
1,N,NaN,NaN,NaN,9783406749360,9.783407e+12,NaN,(ALEPH)020383935EDU01,(DE-465)LT001916077,LT001916077,...,NaN,NaN,NaN,NaN,NaN,NaN,UB23-25962,ger,NaN,NaN
2,N,NaN,NaN,NaN,9783406747298,9.783407e+12,NaN,(ALEPH)020383920EDU01,(DE-465)LT001916062,LT001916062,...,NaN,NaN,NaN,NaN,NaN,NaN,UB23-25962,ger,NaN,NaN
3,N,NaN,NaN,NaN,9783406757808,9.783407e+12,NaN,(ALEPH)020384028EDU01,(DE-465)LT001916170,LT001916170,...,NaN,NaN,NaN,NaN,NaN,NaN,UB23-25962,ger,NaN,NaN
4,N,NaN,NaN,NaN,9783406745843,9.783407e+12,NaN,(ALEPH)020383917EDU01,(DE-465)LT001916059,LT001916059,...,NaN,NaN,NaN,NaN,NaN,NaN,UB23-25962,ger,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4826,N,NaN,NaN,NaN,9783406822100,9.783407e+12,NaN,(Verlag,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,UB23-25962,ger,NaN,NaN
4827,N,NaN,NaN,NaN,9783406823749,9.783407e+12,NaN,(Verlag,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,UB23-25962,ger,NaN,NaN
4828,N,NaN,NaN,NaN,9783406817359,9.783407e+12,NaN,(Verlag,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,UB23-25962,ger,NaN,NaN
4829,N,NaN,NaN,NaN,9783406822445,9.783407e+12,NaN,(Verlag,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,UB23-25962,ger,NaN,NaN


In [5]:
df_alma.columns

Index(['LOCALIZED', 'ISSN', 'ISSN.1', 'ISSN.2', 'ISBN1', 'ISBN2', 'ISBN3',
       'OTHER_SYSTEM_NUMBER1', 'OTHER_SYSTEM_NUMBER2', 'OTHER_SYSTEM_NUMBER3',
       'PORTFOLIO_PID', 'MMS', 'TITLE', 'FROM_YEAR', 'TO_YEAR', 'FROM_MONTH',
       'TO_MONTH', 'FROM_DAY', 'TO_DAY', 'FROM_VOLUME', 'TO_VOLUME',
       'FROM_ISSUE', 'TO_ISSUE', 'PUBLICATION_DATE_OPERATOR',
       'PUBLICATION_DATE_YEAR', 'PUBLICATION_DATE_MONTH', 'GLOBAL_FROM_YEAR',
       'GLOBAL_TO_YEAR', 'GLOBAL_FROM_MONTH', 'GLOBAL_TO_MONTH',
       'GLOBAL_FROM_DAY', 'GLOBAL_TO_DAY', 'GLOBAL_FROM_VOLUME',
       'GLOBAL_TO_VOLUME', 'GLOBAL_FROM_ISSUE', 'GLOBAL_TO_ISSUE',
       'GLOBAL_PUBLICATION_DATE_OPERATOR', 'GLOBAL_PUBLICATION_DATE_YEAR',
       'GLOBAL_PUBLICATION_DATE_MONTH', 'ACCESS_TYPE', 'PERPETUAL_FROM_YEAR',
       'PERPETUAL_TO_YEAR', 'PERPETUAL_FROM_MONTH', 'PERPETUAL_TO_MONTH',
       'PERPETUAL_FROM_DAY', 'PERPETUAL_TO_DAY', 'PERPETUAL_FROM_VOLUME',
       'PERPETUAL_TO_VOLUME', 'PERPETUAL_FROM_ISSUE', 'PERPET

In [8]:
df_alma_sorted = df_alma[['ISBN1', 'TITLE', 'AUTHOR', 'DATE_OF_PUBLICATION', 'URL', 'ELECTRONIC_MATERIAL_TYPE', 'LANGUAGE']]
df_alma_sorted

,ISBN1,TITLE,AUTHOR,DATE_OF_PUBLICATION,URL,ELECTRONIC_MATERIAL_TYPE,LANGUAGE
0,9783406751332,Sir William Siemens Elektronische Ressource 18...,"König, Wolfgang",2020,https://doi.org/10.17104/9783406751356,DOCUMENT,ger
1,9783406749360,Johannes Paul II. Elektronische Ressource Der ...,"Drobinski, Matthias",2020,https://doi.org/10.17104/9783406749384,DOCUMENT,ger
2,9783406747298,Das wichtigste Wissen Elektronische Ressource ...,"Fischer, Ernst Peter",2020,https://doi.org/10.17104/9783406747311,DOCUMENT,ger
3,9783406757808,Polizei- und Ordnungsrecht Elektronische Resso...,"Kingreen, Thorsten",2020,https://doi.org/10.17104/9783406759697,DOCUMENT,ger
4,9783406745843,Allgemeines Schuldrecht Elektronische Ressourc...,"Brox, Hans",2020,https://doi.org/10.17104/9783406747472,DOCUMENT,ger
...,...,...,...,...,...,...,...
4826,9783406822100,Sonderzug nach Moskau Geschichte der deutschen...,"Scianna, Bastian Matteo author",2024,https://doi.org/10.17104/9783406822117,BOOK,ger
4827,9783406823749,"Arbeitsrecht Wilhelm Dütz, Gregor Thüsing","Dütz, Wilhelm author",2024,https://doi.org/10.17104/9783406823756,BOOK,ger
4828,9783406817359,Unter deutscher Besatzung Europa 1939-1945 Tat...,"Tönsmeyer, Tatjana author",2024,https://doi.org/10.17104/9783406817373,BOOK,ger
4829,9783406822445,Geschichte der arabischen Welt Andreas Kaplony,NaN,2024,https://doi.org/10.17104/9783406822452,BOOK,ger


In [11]:
df_alma_doc = df_alma_sorted[df_alma_sorted['ELECTRONIC_MATERIAL_TYPE'] == "DOCUMENT"] 
df_alma_doc

,ISBN1,TITLE,AUTHOR,DATE_OF_PUBLICATION,URL,ELECTRONIC_MATERIAL_TYPE,LANGUAGE
0,9783406751332,Sir William Siemens Elektronische Ressource 18...,"König, Wolfgang",2020,https://doi.org/10.17104/9783406751356,DOCUMENT,ger
1,9783406749360,Johannes Paul II. Elektronische Ressource Der ...,"Drobinski, Matthias",2020,https://doi.org/10.17104/9783406749384,DOCUMENT,ger
2,9783406747298,Das wichtigste Wissen Elektronische Ressource ...,"Fischer, Ernst Peter",2020,https://doi.org/10.17104/9783406747311,DOCUMENT,ger
3,9783406757808,Polizei- und Ordnungsrecht Elektronische Resso...,"Kingreen, Thorsten",2020,https://doi.org/10.17104/9783406759697,DOCUMENT,ger
4,9783406745843,Allgemeines Schuldrecht Elektronische Ressourc...,"Brox, Hans",2020,https://doi.org/10.17104/9783406747472,DOCUMENT,ger
...,...,...,...,...,...,...,...
2998,9783791046099,Buchführung und Bilanzierung in der Sozialwirt...,"Schwien, Bernd",2020,https://doi.org/10.34156/9783791046112,DOCUMENT,ger
2999,9783800636143,Internationale betriebswirtschaftliche Steuerl...,"Scheffler, Wolfram",2009,https://doi.org/10.15358/9783800644063,DOCUMENT,ger
3000,9783800637911,The Innovator's Dilemma Elektronische Ressourc...,"Christensen, Clayton M.",2011,https://doi.org/10.15358/9783800642816,DOCUMENT,ger
3001,9783791042954,Wirtschaftsthemen verständlich vermitteln Elek...,"Hesse, Nils",2019,https://doi.org/10.34156/9783791042978,DOCUMENT,ger


In [12]:
if not pt.started():
    pt.init()

/tmp/ipykernel_3420/3057724015.py:1: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():
Java started and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/tmp/ipykernel_3420/3057724015.py:2: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


In [14]:
#Umbenennung der Spalten zur Vorbereitung auf die Indexierung/Dokumentenrepräsentation
df_alma_doc['docno'] = df_alma_doc['ISBN1']
df_alma_doc['text'] = df_alma_doc['TITLE']
df_alma_doc

/tmp/ipykernel_3420/2028849501.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_alma_doc['docno'] = df_alma_doc['ISBN1']
/tmp/ipykernel_3420/2028849501.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_alma_doc['text'] = df_alma_doc['TITLE']


,ISBN1,TITLE,AUTHOR,DATE_OF_PUBLICATION,URL,ELECTRONIC_MATERIAL_TYPE,LANGUAGE,docno,text
0,9783406751332,Sir William Siemens Elektronische Ressource 18...,"König, Wolfgang",2020,https://doi.org/10.17104/9783406751356,DOCUMENT,ger,9783406751332,Sir William Siemens Elektronische Ressource 18...
1,9783406749360,Johannes Paul II. Elektronische Ressource Der ...,"Drobinski, Matthias",2020,https://doi.org/10.17104/9783406749384,DOCUMENT,ger,9783406749360,Johannes Paul II. Elektronische Ressource Der ...
2,9783406747298,Das wichtigste Wissen Elektronische Ressource ...,"Fischer, Ernst Peter",2020,https://doi.org/10.17104/9783406747311,DOCUMENT,ger,9783406747298,Das wichtigste Wissen Elektronische Ressource ...
3,9783406757808,Polizei- und Ordnungsrecht Elektronische Resso...,"Kingreen, Thorsten",2020,https://doi.org/10.17104/9783406759697,DOCUMENT,ger,9783406757808,Polizei- und Ordnungsrecht Elektronische Resso...
4,9783406745843,Allgemeines Schuldrecht Elektronische Ressourc...,"Brox, Hans",2020,https://doi.org/10.17104/9783406747472,DOCUMENT,ger,9783406745843,Allgemeines Schuldrecht Elektronische Ressourc...
...,...,...,...,...,...,...,...,...,...
2998,9783791046099,Buchführung und Bilanzierung in der Sozialwirt...,"Schwien, Bernd",2020,https://doi.org/10.34156/9783791046112,DOCUMENT,ger,9783791046099,Buchführung und Bilanzierung in der Sozialwirt...
2999,9783800636143,Internationale betriebswirtschaftliche Steuerl...,"Scheffler, Wolfram",2009,https://doi.org/10.15358/9783800644063,DOCUMENT,ger,9783800636143,Internationale betriebswirtschaftliche Steuerl...
3000,9783800637911,The Innovator's Dilemma Elektronische Ressourc...,"Christensen, Clayton M.",2011,https://doi.org/10.15358/9783800642816,DOCUMENT,ger,9783800637911,The Innovator's Dilemma Elektronische Ressourc...
3001,9783791042954,Wirtschaftsthemen verständlich vermitteln Elek...,"Hesse, Nils",2019,https://doi.org/10.34156/9783791042978,DOCUMENT,ger,9783791042954,Wirtschaftsthemen verständlich vermitteln Elek...


In [15]:
df_alma_doc = df_alma_doc[['docno', 'text', 'AUTHOR', 'DATE_OF_PUBLICATION', 'URL', 'ELECTRONIC_MATERIAL_TYPE', 'LANGUAGE']]
df_alma_doc

,docno,text,AUTHOR,DATE_OF_PUBLICATION,URL,ELECTRONIC_MATERIAL_TYPE,LANGUAGE
0,9783406751332,Sir William Siemens Elektronische Ressource 18...,"König, Wolfgang",2020,https://doi.org/10.17104/9783406751356,DOCUMENT,ger
1,9783406749360,Johannes Paul II. Elektronische Ressource Der ...,"Drobinski, Matthias",2020,https://doi.org/10.17104/9783406749384,DOCUMENT,ger
2,9783406747298,Das wichtigste Wissen Elektronische Ressource ...,"Fischer, Ernst Peter",2020,https://doi.org/10.17104/9783406747311,DOCUMENT,ger
3,9783406757808,Polizei- und Ordnungsrecht Elektronische Resso...,"Kingreen, Thorsten",2020,https://doi.org/10.17104/9783406759697,DOCUMENT,ger
4,9783406745843,Allgemeines Schuldrecht Elektronische Ressourc...,"Brox, Hans",2020,https://doi.org/10.17104/9783406747472,DOCUMENT,ger
...,...,...,...,...,...,...,...
2998,9783791046099,Buchführung und Bilanzierung in der Sozialwirt...,"Schwien, Bernd",2020,https://doi.org/10.34156/9783791046112,DOCUMENT,ger
2999,9783800636143,Internationale betriebswirtschaftliche Steuerl...,"Scheffler, Wolfram",2009,https://doi.org/10.15358/9783800644063,DOCUMENT,ger
3000,9783800637911,The Innovator's Dilemma Elektronische Ressourc...,"Christensen, Clayton M.",2011,https://doi.org/10.15358/9783800642816,DOCUMENT,ger
3001,9783791042954,Wirtschaftsthemen verständlich vermitteln Elek...,"Hesse, Nils",2019,https://doi.org/10.34156/9783791042978,DOCUMENT,ger


In [17]:
df_alma_doc = df_alma_doc.rename(columns={"AUTHOR": "Author", "DATE_OF_PUBLICATION": "Pub. Date", "URL": "URL", "ELECTRONIC_MATERIAL_TYPE": "Type", "LANGUAGE": "Language"})
df_alma_doc

,docno,text,Author,Pub. Date,URL,Type,Language
0,9783406751332,Sir William Siemens Elektronische Ressource 18...,"König, Wolfgang",2020,https://doi.org/10.17104/9783406751356,DOCUMENT,ger
1,9783406749360,Johannes Paul II. Elektronische Ressource Der ...,"Drobinski, Matthias",2020,https://doi.org/10.17104/9783406749384,DOCUMENT,ger
2,9783406747298,Das wichtigste Wissen Elektronische Ressource ...,"Fischer, Ernst Peter",2020,https://doi.org/10.17104/9783406747311,DOCUMENT,ger
3,9783406757808,Polizei- und Ordnungsrecht Elektronische Resso...,"Kingreen, Thorsten",2020,https://doi.org/10.17104/9783406759697,DOCUMENT,ger
4,9783406745843,Allgemeines Schuldrecht Elektronische Ressourc...,"Brox, Hans",2020,https://doi.org/10.17104/9783406747472,DOCUMENT,ger
...,...,...,...,...,...,...,...
2998,9783791046099,Buchführung und Bilanzierung in der Sozialwirt...,"Schwien, Bernd",2020,https://doi.org/10.34156/9783791046112,DOCUMENT,ger
2999,9783800636143,Internationale betriebswirtschaftliche Steuerl...,"Scheffler, Wolfram",2009,https://doi.org/10.15358/9783800644063,DOCUMENT,ger
3000,9783800637911,The Innovator's Dilemma Elektronische Ressourc...,"Christensen, Clayton M.",2011,https://doi.org/10.15358/9783800642816,DOCUMENT,ger
3001,9783791042954,Wirtschaftsthemen verständlich vermitteln Elek...,"Hesse, Nils",2019,https://doi.org/10.34156/9783791042978,DOCUMENT,ger


In [18]:
index_folder = "/home/tim/Schreibtisch/2024-25-Data_Librarian-Reschop_Tim/Modul_2-1/alma_index" #anlegen eines Speicherortes für unseres Index/Dokumentenrepräsentation

indexer = pt.DFIndexer(index_folder, overwrite=True) #DataFrameIndexer für die Indexierung auswählen und festlegen wo er gespeichert werden soll und das ein bereits bestehender Index überschrieben werden soll
index_ref = indexer.index(df_alma_doc['text'], df_alma_doc['docno']) #Festlegen, aus welchen Spalten des DF der Index erstellt wird

#Dieser Code erstellt einen IR-Index (Information Retrieval Index) aus dem DataFrame df_bibsonomy. 
#Die Spalte text enthält die Dokumentinhalte, die indexiert werden, und docno wird als eindeutige ID für jedes Dokument verwendet. 
#Mit diesem Index kannst du später Suchanfragen ausführen oder andere IR-Operationen mit PyTerrier durchführen.

print(f"path to our index: {index_ref.toString()}")

/tmp/ipykernel_3420/3376689978.py:3: DeprecationWarning: Call to deprecated class DFIndexer. (use pt.terrier.IterDictIndexer().index(dataframe.to_dict(orient='records')) instead) -- Deprecated since version 0.11.0.
  indexer = pt.DFIndexer(index_folder, overwrite=True) #DataFrameIndexer für die Indexierung auswählen und festlegen wo er gespeichert werden soll und das ein bereits bestehender Index überschrieben werden soll


path to our index: /home/tim/Schreibtisch/2024-25-Data_Librarian-Reschop_Tim/Modul_2-1/alma_index/data.properties


In [19]:
#Index-Instanz aus der zuvor erstellten Index-Referenz (index_ref) erstellen
#pt.IndexFactory: Eine Klasse in PyTerrier, die verwendet wird, um mit Indexen zu arbeiten (z. B. erstellen, laden, verwenden).
#.of(index_ref): Diese Methode erstellt ein Indexobjekt aus der Referenz (index_ref), die auf den Speicherort des Index zeigt.
#Das zurückgegebene Objekt (index) ermöglicht es dir, auf den Index zuzugreifen und Operationen wie Suchen, Abrufen von Metadaten oder Statistiken auszuführen.

index = pt.IndexFactory.of(index_ref)

#Der zuvor erstellte Index (index_ref) ist lediglich ein Verweis auf den Speicherort des Index (normalerweise ein Ordner auf der Festplatte). 
#Mit pt.IndexFactory.of() wird dieser Verweis "geladen" und als voll funktionsfähiger Index bereitgestellt.
#Erst jetzt haben wir einen Index

In [20]:
print(index.getCollectionStatistics().toString())

Number of documents: 3003
Number of terms: 8140
Number of postings: 36033
Number of fields: 0
Number of tokens: 36807
Field names: []
Positions:   false



In [21]:
term = "deutsch" 
index.getLexicon()[term].toString() #Gibt die Ergebnisse für einen einzigen Term aus

#Nt = Anzahl der Dokumente, in denen [term] vorkommt
#TF = Wie oft kommt [term] überhaupt vor
#maxTF = Wie oft kommt [term] in einem Dokument maximal vor

'term177 Nt=56 TF=58 maxTF=2 @{0 10313 5}'

In [23]:
term = "nation" 
index.getLexicon()[term].toString() #Gibt die Ergebnisse für einen einzigen Term aus

'term379 Nt=9 TF=10 maxTF=2 @{0 33948 3}'

In [25]:
#Welcher [term] kommt insgesamt am häufigsten vor? #Herausgabe nach Häufigkeit geordnet

alma_tf_dict = {}  #Erstellen eines Dict. Schlüssel=Term , Wert=Anzahl des Vorkommens

for kv in index.getLexicon():
    alma_tf_dict[kv.getKey()] = kv.getValue().frequency


alma_tf_dict_sorted = sorted(alma_tf_dict.items(), key=lambda x: x[1], reverse=True) #Nicht nach alphabetisch sondern die nach Häufigkeit des Terms ordnen

In [26]:
top_terms = 30 #gibt die 30 häufigstens Terme in gestemmter Form aus
alma_tf_dict_sorted[:top_terms]

[('elektronisch', 3005),
 ('ressourc', 3004),
 ('und', 1092),
 ('der', 975),
 ('die', 716),
 ('von', 558),
 ('geschicht', 380),
 ('ein', 333),
 ('hrsg', 292),
 ('de', 289),
 ('da', 247),
 ('im', 205),
 ('mit', 160),
 ('michael', 158),
 ('hrung', 155),
 ('zur', 144),
 ('bi', 138),
 ('han', 122),
 ('thoma', 114),
 ('andrea', 110),
 ('peter', 108),
 ('einf', 103),
 ('manag', 99),
 ('wolfgang', 98),
 ('christian', 97),
 ('den', 92),
 ('wie', 88),
 ('grundlagen', 83),
 ('unternehmen', 78),
 ('inkl', 77)]

In [29]:
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()

stem = stemmer.stem("Geschichte")
stem


'geschicht'

In [30]:
#wie oft kommt das Gestemmte Wort vor?
lexicon = index.getLexicon()
lexicon[stem].getDocumentFrequency()

375

In [48]:

#Funktion zur BErechnung des IDF-Wertes

def get_idf_for_term(term, index):
    lex = index.getLexicon()
    stemmed_term = stemmer.stem(term)
    if not stemmed_term in lex:
        return

    lex_entry = lex[stemmed_term]
    
    df_term = lex_entry.getDocumentFrequency()
    N = index.getCollectionStatistics().numberOfDocuments
    
    #inverse Dokumentenfrequenz
    idf = N/df_term
    #logarithmus, damit die Relevanz nicht proportional berechnet wird
    log_idf = np.emath.logn(10, idf)
    
    return log_idf
    


In [47]:
get_idf_for_term('deutsch', index)

1.7293673051927805

In [ ]:
#TF-IDF Funktion

def calc_tf_idf(query, docno, index):

In [49]:
search_engine_tfidf = pt.BatchRetrieve(index, wmodel="TF_IDF")

/tmp/ipykernel_3420/2880449899.py:1: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  search_engine_tfidf = pt.BatchRetrieve(index, wmodel="TF_IDF")


In [50]:
search_engine_tfidf.search("Geschichte")

,qid,docid,docno,rank,score,query
0,1,630,9783406638152,0,2.186318,Geschichte
1,1,748,9783406640544,1,2.186318,Geschichte
2,1,1189,9783406712920,2,2.186318,Geschichte
3,1,1250,9783406715136,3,2.186318,Geschichte
4,1,1375,9783406470769,4,2.186318,Geschichte
...,...,...,...,...,...,...
370,1,1434,9783406313493,370,1.213579,Geschichte
371,1,1473,9783406492754,371,1.213579,Geschichte
372,1,2165,9783406613531,372,1.213579,Geschichte
373,1,1530,9783406509254,373,1.185811,Geschichte


In [51]:
alma_dict = df_alma_doc.to_dict(orient='records')
alma_dict

[{'docno': '9783406751332',
  'text': 'Sir William Siemens Elektronische Ressource 1823-1883 Wolfgang König',
  'Author': 'König, Wolfgang',
  'Pub. Date': 2020,
  'URL': 'https://doi.org/10.17104/9783406751356',
  'Type': 'DOCUMENT',
  'Language': 'ger'},
 {'docno': '9783406749360',
  'text': 'Johannes Paul II. Elektronische Ressource Der Papst, der aus dem Osten kam Matthias Drobinski, Thomas Urban',
  'Author': 'Drobinski, Matthias',
  'Pub. Date': 2020,
  'URL': 'https://doi.org/10.17104/9783406749384',
  'Type': 'DOCUMENT',
  'Language': 'ger'},
 {'docno': '9783406747298',
  'text': 'Das wichtigste Wissen Elektronische Ressource Vom Urknall bis heute Ernst Peter Fischer',
  'Author': 'Fischer, Ernst Peter',
  'Pub. Date': 2020,
  'URL': 'https://doi.org/10.17104/9783406747311',
  'Type': 'DOCUMENT',
  'Language': 'ger'},
 {'docno': '9783406757808',
  'text': 'Polizei- und Ordnungsrecht Elektronische Ressource mit Versammlungsrecht Thorsten Kingreen, Ralf Poscher, Bodo Pieroth, Ber

In [52]:
index_folder_alma = "/home/tim/Schreibtisch/2024-25-Data_Librarian-Reschop_Tim/Modul_2-1/alma_index"

fields=['docno', 'text', 'Author', 'Pub. Date', 'URL', 'Type', 'Language']

indexer_alma = pt.IterDictIndexer(index_folder_alma, meta={'docno': 50, 'text': 200}, fields=True, text_attrs=fields, overwrite=True)
index_ref_alma = indexer_alma.index(alma_dict)

print(f"path to our index: {index_ref_alma.toString()}")

path to our index: /home/tim/Schreibtisch/2024-25-Data_Librarian-Reschop_Tim/Modul_2-1/alma_index/data.properties


In [54]:
index_alma = pt.IndexFactory.of(index_ref_alma)
print(index_alma.getCollectionStatistics().toString())

Number of documents: 3003
Number of terms: 8140
Number of postings: 36034
Number of fields: 7
Number of tokens: 36808
Field names: [docno, text, Author, Pub. Date, URL, Type, Language]
Positions:   false



In [55]:
search_engine_alma = pt.terrier.Retriever(index_alma, wmodel="TF_IDF")

In [56]:
res = search_engine_alma.search("deutsch")
res

,qid,docid,docno,rank,score,query
0,1,691,9783406656309,0,4.162160,deutsch
1,1,70,9783406736063,1,3.979053,deutsch
2,1,1195,9783406729584,2,3.979053,deutsch
3,1,1250,9783406715136,3,3.979053,deutsch
4,1,2140,9783406562488,4,3.979053,deutsch
5,1,2239,9783406536281,5,3.817988,deutsch
6,1,1992,9783648120255,6,3.675582,deutsch
7,1,20,9783406754791,7,3.669455,deutsch
8,1,1752,9783406480072,8,3.669455,deutsch
9,1,471,9783406661068,9,3.532046,deutsch


In [58]:
pickle.dump(df_alma_doc, open("/home/tim/Schreibtisch/2024-25-Data_Librarian-Reschop_Tim/Modul_2-1/alma_index/alma_beck.pkl", "wb"))